In [116]:
import time

import pygame, sys, copy, math
import random

În Blackjack, fiecare carte are o anumită valoare.
Cărțile de la 2 la 10 au chiar valoarea înscrisă pe ele.
J (valet), Q (regină), K (rege): valorează 10 puncte fiecare.
A (asul): poate valora 1 sau 11 puncte, în funcție de ce e în avantajul jucătorului.

Desfășurarea Jocului
Dealerul împarte câte două cărți fiecărui jucător (de regulă cu fața în sus) și două cărți pentru sine (una cu fața în sus, și una cu fața în jos).
Opțiunile jucătorului:
Hit: Mai trage o carte (modificând scorul).
Stand: Se oprește și îngheață scorul, transferând controlul către Dealer.


După ce jucătorul s-a oprit, dealerul își arată cartea ascunsă. Spre deosebire de jucători, dealerul nu are voie să aleagă strategic ci după regulile:
- Dacă are 16 puncte sau mai puțin, este obligat să tragă carte (Hit).
- Dacă are 17 puncte sau mai mult, este obligat să se oprească (Stand).

Terminarea jocului:
Pierdere: Dacă jucătorul depășește 21 (chiar dacă și dealerul are peste 21).
Câștig: 
- Caz 1: dacă scorul jucătorului este mai mare decât al dealerului 
- Caz 2: dacă dealerul sare de 21 (dar jucătorul nu).
Remiză (Egalitate): Dacă jucătorul și dealerul au același punctaj.

In [ ]:
ADANCIME_MAX=4

def scor(carti):
    scor=0
    for elem in carti:
        if type(elem) == str:
            scor+=InfoJoc.SCORURI[elem]
        else:
            scor+=elem
    asi = carti.count("A")        
    # Dacă scorul e peste 21 și avem ași socotiți ca 11, 
    # îi transformăm în 1 pe rând
    while scor> 21 and asi > 0:
        scor-= 10
        asi-= 1
    return scor

class InfoJoc:
    """
    Clasa care defineste jocul. Se va schimba de la un joc la altul.
    """
    CARTI=(list(range(2,11))+["A","J","Q","K"])*4
    SCORURI={
        "A":11,
        "J":10,
        "Q":10,
        "K":10
    }

    
    def __init__(self, dealer=None, jucator=None, carti=None, continua=None):
        if carti:
            self.carti=carti
            self.dealer=dealer
            self.jucator=jucator
            self.continua=continua
        else:
            #starea initiala
            self.dealer=random.choices(population=InfoJoc.CARTI,k=2)
            self.carti=list(InfoJoc.CARTI)
            self.carti.remove(self.dealer[0]) #consideram aceasta cartea ascunsa
            self.carti.remove(self.dealer[1])
            self.jucator=random.choices(population=InfoJoc.CARTI,k=2)
            self.carti.remove(self.jucator[0])
            self.carti.remove(self.jucator[1])
            self.continua=True

    
    def final(self):
        if scor(self.jucator)>21:
            return "dealer"
        if not self.continua:
            if scor(self.dealer)==scor(self.jucator):
                return "remiza"
            if scor(self.jucator)>scor(self.dealer):
                return "jucator"
            else:
                return "dealer"
        if scor(self.dealer)>21:
            return "jucator"
        return False

    def mutari(self, jucator):
        lMutari=[]
        if self.continua:
            #muta jucatorul
            mutareStand=InfoJoc(
                dealer=self.dealer,
                jucator=self.jucator,
                carti=self.carti,
                continua=False)
            mutareHit=InfoJoc(
                dealer=self.dealer,
                jucator=self.jucator,
                carti=self.carti,
                continua=True)   
        else:
            return [self]
                    
        return [mutareStand, mutareHit]

    #restAdancime = cat mai are pana ajunge la adancimea maxima    
    def estimeaza_scor(self, restAdancime):
        if self.final()== InfoJoc.JMAX:
            return 100+restAdancime
        elif self.final()== InfoJoc.JMIN:
            return -100-restAdancime
        elif self.final()=="remiza":
            return 0
        else:
            if self.continua:
                return scor(self.jucator)-(2+scor(self.dealer[1:]))
            return scor(self.jucator)-scor(self.dealer)

    def sirAfisare(self):
        sir="Dealer: "
        if self.continua:
            sir+="carte_ascunsa, "+", ".join(map(str,self.dealer[1:]))+"\n"
        else:
            sir+=", ".join(map(str,self.dealer))+"\n"
        sir+="Jucator: "+", ".join(map(str,self.jucator))+"\n"
        return sir

    def __str__(self):
        return self.sirAfisare()

    def __repr__(self):
        return self.sirAfisare() 

Clasa corespunzatoare starilor jocului

In [118]:
class Stare:
    """
    Clasa folosita de algoritmii minimax si alpha-beta
    Are ca proprietate tabla de joc
    Functioneaza cu conditia ca in cadrul clasei InfoJoc sa fie definiti JMIN si JMAX (cei doi jucatori posibili)
    De asemenea cere ca in clasa InfoJoc sa fie definita si o metoda numita mutari() care ofera lista cu configuratiile posibile in urma mutarii unui jucator
    """
    def __init__(self, tabla_joc, j_curent, restAdancime, parinte=None, estimare=None):
        self.tabla_joc=tabla_joc
        self.j_curent=j_curent
        
        #restul de adancime in arborele de stari pana la adancimea maxima
        self.restAdancime=restAdancime    
        
        #estimarea favorabilitatii starii (daca e finala) sau al celei mai bune stari-fiice (pentru jucatorul curent)
        self.estimare=estimare
        
        #lista de mutari posibile din starea curenta
        self.mutari_posibile=[]
        
        #cea mai buna mutare din lista de mutari posibile pentru jucatorul curent
        self.stare_aleasa=None


    def mutari(self):        
        l_mutari=self.tabla_joc.mutari(self.j_curent)
        l_stari_mutari=[Stare(mutare, self.j_curent, self.restAdancime-1, parinte=self) for mutare in l_mutari]

        return l_stari_mutari
        
    
    def __str__(self):
        sir= str(self.tabla_joc) + "(Juc curent:"+self.j_curent+")\n"
        return sir

Algoritmul ExpectiMax

In [119]:


def expectimax(stare):
    if stare.tabla_joc.final() or stare.restAdancime==0:
        stare.estimare= stare.tabla_joc.estimeaza_scor(stare.restAdancime)
        stare.stare_aleasa=stare
        return stare
    
    mutari= [expectimax(mutare) for mutare in stare.mutari()]
    if stare.j_curent==InfoJoc.JMAX:
        stare.stare_aleasa=max(mutari, key=lambda x: x.estimare )
        stare.estimare=stare.stare_aleasa.estimare
    else: # este nod aleator
        stare.estimare=0
        stare.stare_aleasa=stare #nu aleg stare fiind nod aleator
        jocCurent=stare.tabla_joc
        if jocCurent.continua:
            for carte in jocCurent.carti:
                cartiJucator=jocCurent.jucator+[carte]
                cartiNoi=list(jocCurent.carti)
                cartiNoi.remove(carte)
                mutareNoua=InfoJoc(
                    dealer=jocCurent.dealer, 
                    jucator=cartiJucator,
                    carti=cartiNoi,
                    continua=True)
                stare.estimare+=probCarte*expectimax(
                    Stare(mutareNoua, 
                        stare.j_curent, 
                        stare.restAdancime-1, 
                        parinte=stare)).estimare
        else:
            for carte in jocCurent.carti:
                probCarte=1/len(jocCurent.carti)
                cartiDealer=jocCurent.jucator+[carte]
                cartiNoi=list(jocCurent.carti)
                cartiNoi.remove(carte)
                mutareNoua=InfoJoc(
                    dealer=cartiDealer, 
                    jucator=jocCurent.jucator,
                    carti=cartiNoi,
                    continua=False)
                stare.estimare+=probCarte*expectimax(mutareNoua).estimare
        
    
    return stare

In [120]:
def afis_daca_final(stare_curenta):
    final=stare_curenta.tabla_joc.final()
    if(final):
        if (final=="remiza"):
            print("Remiza!")
        else:
            print("A castigat "+final)
            
        return True
        
    return False
        
    

def main():
    # raspuns_valid=False
    # while not raspuns_valid:
    #     InfoJoc.JMIN=input("Doriti sa fiti dealer sau jucator? ").lower()
    #     if (InfoJoc.JMIN in ['dealer', 'jucator']):
    #         raspuns_valid=True
    #     else:
    #         print("Raspunsul trebuie sa fie x sau 0.")
    # InfoJoc.JMAX= 'dealer' if InfoJoc.JMIN == 'jucator' else 'dealer'
    
    #deoarece dealerul nu face alegeri, nu e interesant ca AI
    InfoJoc.JMAX= 'jucator'
    InfoJoc.JMIN= 'dealer'
    
    #initializare tabla
    tabla_curenta=InfoJoc()
    print("Tabla initiala")
    print(str(tabla_curenta))
    
    #creare stare initiala
    stare_curenta=Stare(tabla_curenta,'jucator',ADANCIME_MAX)
    

    while True :
        

        if not stare_curenta.tabla_joc.continua:
            while scor(stare_curenta.tabla_joc.dealer) <= 16:
                carteAleasa=random.choice(stare_curenta.tabla_joc.carti)
                stare_curenta.tabla_joc.carti.remove(carteAleasa)
                stare_curenta.tabla_joc.dealer.append(carteAleasa)
                print("Dealerul a primit cartea ", carteAleasa)
            afis_daca_final(stare_curenta)
            break
            
        #--------------------------------
        else: #jucatorul calculatorul
            t_inainte=int(round(time.time() * 1000))
            stare_actualizata=expectimax(stare_curenta)
            stare_curenta.tabla_joc=stare_actualizata.stare_aleasa.tabla_joc
            print("Tabla dupa mutarea calculatorului")
            print(str(stare_curenta))
            t_dupa=int(round(time.time() * 1000))
            print("Calculatorul a \"gandit\" timp de "+str(t_dupa-t_inainte)+" milisecunde.")
            if stare_curenta.tabla_joc.continua:
                print("Hit")
                carteAleasa=random.choice(stare_curenta.tabla_joc.carti)
                stare_curenta.tabla_joc.carti.remove(carteAleasa)
                stare_curenta.tabla_joc.jucator.append(carteAleasa)
                print("Jucatorul a primit cartea ", carteAleasa)
                print("Scor jucator:", scor(stare_curenta.tabla_joc.jucator))
            else:
                print("Stand!")
            if (afis_daca_final(stare_curenta)):
                break
            
if __name__ == "__main__" :
    main()


Tabla initiala
Dealer: carte_ascunsa, 9
Jucator: 9, A

Tabla dupa mutarea calculatorului
Dealer: 5, 9
Jucator: 9, A
(Juc curent:jucator)

Calculatorul a "gandit" timp de 0 milisecunde.
Stand!
A castigat jucator
